In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes', 'safetensors']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'safetensors': ('0.8.0', 'safetensors==0.8.0', 'safetensors==0.8.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Saving, Loading, and Pretrained Weights

A trained network is two separate things kept in two separate places. The
*code* is the class you wrote: its layers, its `forward` pass, the config that
sized it. The *model state* is the tensors that training filled in: weights,
biases, and running statistics. Optimizer moments, random-number streams, and
the step belong to the wider *training state*. A weight file saves model state;
a resumable checkpoint saves full training state. The code
stays in your source repository, under version control, exactly like any other
Python. To bring a model back you need both halves: run the code to rebuild an
empty network, then pour the saved state into it.

This split explains most of what follows. It is why a checkpoint cannot
resurrect a model on its own, why the config object from
that section belongs *inside* the checkpoint, and why the
format that stores the state matters once you start sharing files with people who
do not have your code.

In [ ]:
import json
import struct
import warnings
from dataclasses import asdict, dataclass
import numpy as np
from d2l import tensorflow as d2l
import tensorflow as tf
from safetensors.tensorflow import load_file, save_file

## State, Not Code

The state of a network is a collection of named variables, the weights of
that section. Before we save a whole model, the warm-up is that
`np.save` and `np.load` work on any tensor, and, through NumPy's pickle
fallback, on the dicts that hold them.

In [ ]:
x = tf.range(4)
np.save('tensors.npy', {'x': x, 'y': tf.zeros(4)})
np.load('tensors.npy', allow_pickle=True).item()

A model's state is one such collection, built for you: `net.weights` is the
list of its variables, and each variable's `path` names its place in the layer
tree (`mlp/hidden/kernel`, `mlp/output/bias`; Keras calls a weight matrix a
`kernel`). Keras invents layer names like `dense_3` when you do not choose
them, so we name the layers explicitly to keep the paths stable across
instances. Here is the tree for a small MLP.

In [ ]:
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__(name='mlp')
        self.hidden = tf.keras.layers.Dense(256, name='hidden')
        self.out = tf.keras.layers.Dense(10, name='output')

    def call(self, x):
        return self.out(tf.nn.relu(self.hidden(x)))

net = MLP()
X = tf.random.normal((2, 20))
Y = net(X)
{v.path: tuple(v.shape) for v in net.weights}

Nothing in this dictionary knows it came from a class called `MLP`. The names
and shapes refill a network with a compatible architecture and naming
structure. Renaming an attribute or changing the nesting can break that match
even when the computation is equivalent; the final section shows how to
migrate such keys explicitly.

## safetensors: the Interchange Format

`np.save` writes NumPy's `.npy` format. For a single tensor that is pure data:
a small header with the dtype and shape, then the raw bytes. The warm-up dict
is another matter. NumPy can only store a dict by falling back to Python's
`pickle`, which does not store data so much as a program that *reconstructs*
data, and loading runs that program; that is what `allow_pickle=True` opted
into. For a file you wrote and never let out of your control this is harmless.
For a file you downloaded it is a remote-code-execution vector: any object in
the stream can name a callable for the loader to invoke, which is why NumPy
refuses pickled contents by default (`allow_pickle=False`).

TensorFlow's own model files sidestep pickle: neither a TF checkpoint nor the
Keras `.weights.h5`/`.keras` formats execute it on load. They are not
automatically safe to download, though. A `.keras` archive can carry serialized
code (a `Lambda` layer's function), which is why Keras refuses to deserialize
such code unless you pass `safe_mode=False`, and why a model file from an
untrusted source deserves the same caution as a pickle.

safetensors removes the problem at the root by
having no program to run. As the figure lays out byte
by byte, a safetensors file is an 8-byte little-endian integer giving the
header length, then that many bytes of JSON naming each tensor's dtype, shape,
and byte range, then the raw tensor bytes back to back. There is no opcode
stream to interpret, so the format itself carries no executable program.
Loaders must still validate malformed headers, offsets, and allocation sizes.
It is also framework-neutral and memory-mappable, which is why model hubs
commonly prefer it.
Save and reload the MLP's state through it and confirm the round trip is exact.

![The safetensors file as one horizontal byte strip: an 8-byte header length, a JSON header naming each tensor's dtype, shape, and byte offsets, and the raw tensor bytes packed back to back with no gaps, with two of the file's own data_offsets entries traced down to their exact span in the bar.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-safetensors-layout.svg)

Here the variable paths do the naming, so the flat mapping is a one-line
comprehension over `net.weights`. Two quirks of the `tensorflow` binding to
know: it converts through NumPy in both directions, so `load_file` hands back
constant tensors that you `assign` into a model's variables, and `save_file`
overwrites the values of the dict you pass it during that conversion, so give
it a throwaway copy.

In [ ]:
state = {v.path: v for v in net.weights}
save_file(dict(state), 'mlp-tf.safetensors')  # dict(): save_file clobbers its arg
restored = load_file('mlp-tf.safetensors')
clone = MLP()
clone(X)                                   # build the variables first
for v in clone.weights:
    v.assign(restored[v.path])
tf.reduce_all(clone(X) == Y)

Because the header is plain JSON at a known offset, you can read it without the
library and see there is no magic to the format.

In [ ]:
with open('mlp-tf.safetensors', 'rb') as f:
    n = struct.unpack('<Q', f.read(8))[0]   # header length, little-endian
    header = json.loads(f.read(n))
header['mlp/hidden/kernel']

`np.save` and Keras's own `.weights.h5` keep their place for your own scratch
files and checkpoints. safetensors is what you use to hand a model to anyone
else.

## Checkpointing a Training Run

A checkpoint you can resume from holds more than weights. Resuming means picking
up the optimizer where it stopped, and Adam's state is the running first and
second moments of the gradients from that section. Drop them and
the optimizer restarts its momentum from zero, so the first steps after a resume
no longer behave like a continuation. A full checkpoint therefore bundles the
model state, the optimizer state, the RNG state (so data shuffling and dropout
continue the same stream), the step counter, and the config that sizes the model
when you rebuild it. the figure pairs each of those
five compartments with the exact thing it restores on resume.

![A checkpoint file's five compartments, each paired by an arrow with what it restores on resume: model state_dict with weights, optimizer state with momentum and second moments, RNG state with data order and dropout, step with schedule position, and config with architecture.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-checkpoint-contents.svg)

In TensorFlow the bundle already has a name. `tf.train.Checkpoint` takes the
objects to track as keyword arguments, model, optimizer, and a step counter,
walks their variables, and saves and restores them as one unit, Adam's moment
estimates included. Because this native already covers the job, the tensorflow
tab defines no helper of its own; the calls below are the idiom as you would
write it in any project. Saves are numbered (`run-tf-1`, `run-tf-2`, ...), so a
crash cannot corrupt the previous completed number. The config travels in a
JSON sidecar named for that number. A tracked `tf.random.Generator` carries the
random stream; the process-global generator is not captured automatically.

Train a tiny regressor for a hundred steps and checkpoint it. The `Config`
dataclass is what a rebuild reads to size the model, so it travels with the
weights.

In [ ]:
@dataclass
class Config:
    in_dim: int = 20
    hidden: int = 64
    lr: float = 0.05

def build(cfg):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(cfg.hidden, activation='relu'),
        tf.keras.layers.Dense(1)])

tf.keras.utils.set_random_seed(1)
cfg = Config()
data = tf.random.normal((256, cfg.in_dim))
target = data @ tf.random.normal((cfg.in_dim, 1)) + 0.1 * tf.random.normal((256, 1))
loss = tf.keras.losses.MeanSquaredError()

def step(model, opt):
    with tf.GradientTape() as tape:
        l = loss(target, model(data))
    opt.apply_gradients(zip(tape.gradient(l, model.trainable_variables),
                            model.trainable_variables))
    return float(l)

net = build(cfg)
net(data[:1])                                # build the variables
opt = tf.keras.optimizers.Adam(cfg.lr)
for _ in range(100):
    step(net, opt)

rng = tf.random.Generator.from_seed(1)
ckpt = tf.train.Checkpoint(model=net, optimizer=opt, step=tf.Variable(100),
                           rng=rng)
path = ckpt.save('run-tf')                   # a numbered save: 'run-tf-1'
with open(path + '.cfg.json', 'w') as f:
    json.dump(asdict(cfg), f)
path, round(float(loss(target, net(data))), 4)

The restore is exact. Corrupt every parameter, load the checkpoint back, and the
loss returns to where it was.

Note the shape of the restore call: `restore` patches the tracked objects in
place, matching each variable by its route through the object graph (`model`,
then the layer, then its `kernel`) rather than by name, and it returns a status
object. `assert_consumed()` on that status checks that every saved value found
a variable and every variable found a value, turning a silent partial restore
into a loud error.

In [ ]:
for v in net.trainable_variables:
    v.assign_add(tf.ones_like(v))         # wreck the weights
before = float(loss(target, net(data)))
ckpt.restore(path).assert_consumed()
after = float(loss(target, net(data)))
f'perturbed {before:.2f} -> restored {after:.4f}'

Now the reason the optimizer state is in the file. Resume the run two ways from
the same checkpoint: once restoring the optimizer, once with a fresh one holding
only the weights. The network is near its minimum, so the correct continuation
barely moves. A fresh Adam, with its moment estimates reset and its bias
correction starting over, takes an oversized first step and overshoots.

One Keras 3 tripwire sits in the resume path. A freshly constructed Adam owns
no slot variables; the moment estimates are created only when the optimizer is
built. Restore into a fresh optimizer without building it first and the saved
moments have no variables to land in, so `assert_consumed()` fails with an
unresolved `optimizer` object. The fix is one line before the restore:
`opt.build(model.trainable_variables)`.

In [ ]:
with open(path + '.cfg.json') as f:
    cfg = Config(**json.load(f))
net_full = build(cfg)
net_full(data[:1])
opt_full = tf.keras.optimizers.Adam(cfg.lr)
opt_full.build(net_full.trainable_variables)   # create Adam's slots first
rng_full = tf.random.Generator.from_seed(0)
tf.train.Checkpoint(model=net_full, optimizer=opt_full,
                    step=tf.Variable(0), rng=rng_full).restore(path).assert_consumed()
full = [round(step(net_full, opt_full), 4) for _ in range(5)]

net_fresh = build(cfg)
net_fresh(data[:1])
tf.train.Checkpoint(model=net_fresh).restore(path).expect_partial()  # weights only
opt_fresh = tf.keras.optimizers.Adam(cfg.lr)
fresh = [round(step(net_fresh, opt_fresh), 4) for _ in range(5)]

print('full  optimizer:', full)
print('fresh optimizer:', fresh)

The full-state run keeps descending; the weights-only run spikes and has to claw
its way back. That transient is the cost of forgetting the optimizer, and it is
why "just the weights" is not a resumable checkpoint.

For models too large to hold in memory, the format already cooperates: a TF
checkpoint is an index file plus data shards rather than one monolith, and
restores are lazy, so a variable created later is filled from the file at
creation time instead of everything materializing up front. Multi-host
`tf.distribute` jobs build on the same machinery;
that section returns to it when models get that big.

## Loading Weights You Did Not Train

The most common reason to load weights is that someone else produced them. You
take a network trained on a large dataset and adapt it: keep the learned
feature extractor, replace the final layer for your own labels.
`keras.applications` is the built-in zoo; `weights='imagenet'` downloads the
matching parameters the first time, and `include_top=False` drops the
1000-class head so you can attach your own.

In [ ]:
backbone = tf.keras.applications.MobileNetV2(   # ~9 MB on first run
    weights='imagenet', include_top=False, input_shape=(160, 160, 3),
    pooling='avg')
net = tf.keras.Sequential([backbone, tf.keras.layers.Dense(10, name='head')])
net(tf.zeros((1, 160, 160, 3)))                 # build the new 10-class head
net.layers[-1]

Keras loads weight files whole, so the partial-loading control is a flag rather
than dict surgery: `load_weights(path, skip_mismatch=True)` fills every
variable whose saved shape matches and skips the rest, reporting the skips as a
warning. To stage a mismatch, save the weights of a donor whose head has 101
classes, our stand-in for a fine-tuned model someone else published, then load
that file into the 10-class network. The cell records the warning so we can
read it back as a report.

In [ ]:
donor = tf.keras.Sequential([backbone, tf.keras.layers.Dense(101, name='head')])
donor(tf.zeros((1, 160, 160, 3)))
donor.save_weights('donor-101.weights.h5')   # stand-in for a downloaded file

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    net.load_weights('donor-101.weights.h5', skip_mismatch=True)
report = str(caught[-1].message).splitlines()
print(report[0])
print(report[2].split(' Target variable:')[0])

Read the warning; do not silence it. It names exactly one object that could not
be loaded, the `head` layer, and gives the two shapes that disagree, `(1280,
10)` against `(1280, 101)`. That is the expected mismatch: the head is new and
meant to start random. Any other layer in that list would mean the backbones
disagree, a wrong input shape or a renamed layer, and is a bug rather than
something to skip. One piece of API drift to know: older tutorials pass
`by_name=True` for partial loads, but in Keras 3 that flag only applies to
legacy `.h5` files and *raises* on the native `.weights.h5` format;
`skip_mismatch` is the current control.

In [ ]:
backbone.trainable = False
trainable = sum(int(tf.size(v)) for v in net.trainable_variables)
total = sum(int(tf.size(v)) for v in net.weights)
f'{trainable} trainable of {total}'

`keras.applications` is one source; the Hugging Face Hub is the ecosystem-scale
one, and it distributes its weights as safetensors, which closes the loop with
the format of the previous section. This section covers *how* to load and adapt
pretrained weights; that section covers when it helps and how far
to unfreeze.

## Summary

A saved model is state, not code: a collection of path-named variables that
means something only once the code that built the network runs again. For your
own files `np.save` and `.weights.h5` are fine; for files you share,
safetensors stores the same tensors behind a plain JSON header, with no pickle
to execute, which is why hubs standardize on it. A resumable checkpoint is a
`tf.train.Checkpoint` of model, optimizer, and step saved as one unit, or a
resume restarts the optimizer's momentum from zero; a fresh optimizer must be
built before the restore so Adam's moments have somewhere to land. Loading
someone else's weights is `load_weights` with `skip_mismatch=True`, and the
skip warning is a diagnostic to read rather than a message to silence.

## Exercises

1. Even if you never deploy to another machine, name two reasons to checkpoint.
   Then consider the atomic write: if the checkpoint were written straight to its
   final path (delete the `os.replace` from `save_checkpoint`, the
   rename-into-place that orbax performs, or the fresh numbered files that
   `tf.train.Checkpoint.save` writes), describe the failure a crash mid-write
   now causes, and why the atomic version avoids it.
1. Read the first 8 bytes of the safetensors file you wrote for the MLP as a
   little-endian integer, as the header cell does. How large is the JSON header
   for the MLP, and how does it grow if you double the hidden width?
1. Save the MLP's parameters cast to `bfloat16` and load them back into a
   `float32` model (that section). What is lost? Is that acceptable
   for inference? For resuming training?
1. Take two checkpoints of the regressor 50 steps apart, average their weight
   tensors into a third set of parameters, and evaluate it. The result previews
   weight averaging.